A modernized Learning Rate Range Test script, designed to be a companion
to the main training pipeline. It finds an optimal learning rate by running
a short, mock training session where the LR is increased exponentially.

1. Set Up Environment

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    texlive-latex-base \
    texlive-latex-recommended \
    texlive-latex-extra \
    texlive-fonts-recommended \
    texlive-fonts-extra

In [ ]:
!pdflatex --version

In [ ]:
# --- 1. Environment Setup ---
print("Installing required packages...")
!pip install -q torch-lr-finder albumentations

In [ ]:
!pip install git+https://github.com/qubvel/segmentation_models.pytorch

2. Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import shutil
import zipfile
import warnings
import json
import re
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
import pandas as pd
from datetime import datetime
import gc
import time

import math
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import matplotlib
matplotlib.use("Agg")

In [ ]:
try:
    from torch_lr_finder import LRFinder
except ImportError:
    print("torch-lr-finder not found. Please ensure it's installed.")
    !pip install -q torch-lr-finder
    try:
         from torch_lr_finder import LRFinder
    except ImportError:
         raise ImportError("Failed to import LRFinder even after install attempt.")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
def change_paths(path, possible_paths=None):
    if possible_paths is None:
        possible_paths = ['/content/drive/MyDrive/Personal_Drive_Bruno/','/content/drive/MyDrive/']
    for p in possible_paths:
      path=os.path.join(p,path)
      if os.path.exists(path):
        return path
    raise ValueError(f"The path {path} does not match any known environments.")

In [ ]:
# -----------------------------
# Reproducibility
# -----------------------------
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
class BCEDiceHybridLossPaper(nn.Module):
    """
    Paper-faithful implementation of the hybrid loss from:

    Khened, M., Kori, A., Rajkumar, H. et al.
    A generalized deep learning framework for whole-slide image segmentation and analysis.
    Scientific Reports 11, 11579 (2021).
    https://doi.org/10.1038/s41598-021-90444-8

    Loss = alpha * CE + beta * Dice_BG + gamma * Dice_FG

    - CE is binary cross-entropy on the tumor posterior p_i
    - Dice uses squared denominator: sum(p^2) + sum(g^2)
    """

    def __init__(self,
                 alpha: float = 0.5,
                 beta: float = 0.25,
                 gamma: float = 0.25,
                 smooth: float = 1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    @staticmethod
    def _flatten(x):
        # [B,H,W] -> [B,N]
        return x.reshape(x.size(0), -1)

    def _dice_loss_khened(self, p, g):
        """
        p, g: [B,H,W] probabilities and binary GT for ONE class
        Implements:
            DL = 1 - (2 sum(p g)) / (sum(p^2) + sum(g^2))
        """
        p = self._flatten(p)
        g = self._flatten(g)

        intersection = (p * g).sum(dim=1)
        denom = (p.pow(2).sum(dim=1) + g.pow(2).sum(dim=1))

        dice = (2.0 * intersection + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()

    def forward(self, logits, target_one_hot):
        """
        logits:          [B,2,H,W] raw model outputs
        target_one_hot:  [B,2,H,W] one-hot encoded masks
        """

        if logits.shape != target_one_hot.shape:
            raise ValueError(
                f"Shape mismatch: logits {logits.shape}, target {target_one_hot.shape}"
            )

        # -------------------------------------------------
        # 1) Posterior probabilities (softmax)
        # -------------------------------------------------
        probs = torch.softmax(logits, dim=1)

        # Tumor posterior p_i and GT g_i (paper notation)
        p_fg = probs[:, 1, :, :]
        g_fg = target_one_hot[:, 1, :, :]

        p_fg = p_fg.clamp(1e-7, 1.0 - 1e-7)

        # -------------------------------------------------
        # 2) Binary Cross-Entropy (Eq. 2)
        # -------------------------------------------------
        ce_loss = -(g_fg * torch.log(p_fg) +
                    (1.0 - g_fg) * torch.log(1.0 - p_fg))
        ce_loss = ce_loss.mean()

        # -------------------------------------------------
        # 3) Dice losses (Eq. 1)
        # -------------------------------------------------
        # Background
        dice_bg = self._dice_loss_khened(
            probs[:, 0, :, :],
            target_one_hot[:, 0, :, :]
        )

        # Foreground (tumor)
        dice_fg = self._dice_loss_khened(
            probs[:, 1, :, :],
            target_one_hot[:, 1, :, :]
        )

        # -------------------------------------------------
        # 4) Hybrid combination (Eq. 3)
        # -------------------------------------------------
        loss = (
            self.alpha * ce_loss +
            self.beta  * dice_bg +
            self.gamma * dice_fg
        )

        return loss

In [ ]:
# -----------------------------
# Dataset (minimal; adapt to your paths)
# -----------------------------
class ProstateCancerDataset(Dataset):
    """
    Minimal dataset consistent with your training script structure:
    TRAIN/CANCER, TRAIN/CANCER_MASK, TRAIN/NOT_CANCER, TRAIN/NOT_CANCER_MASK, etc.
    File naming must include PATIENT_<id>_ for patient-level stratification.
    """
    def __init__(
        self,
        cancer_image_dir: str,
        cancer_mask_dir: str,
        not_cancer_image_dir: str,
        not_cancer_mask_dir: str,
        mean: Optional[List[float]] = None,
        std: Optional[List[float]] = None,
        compute_stats: bool = False,
        stats_sample_size: Optional[int] = None,
    ):
        self.cancer_image_dir = cancer_image_dir
        self.cancer_mask_dir = cancer_mask_dir
        self.not_cancer_image_dir = not_cancer_image_dir
        self.not_cancer_mask_dir = not_cancer_mask_dir

        self.image_paths: List[str] = []
        self.mask_paths: List[str] = []
        self.labels: List[int] = []
        self.patient_ids: List[str] = []

        patient_id_pattern = re.compile(r"PATIENT_(\d+)_")

        cancer_images = [f for f in os.listdir(cancer_image_dir) if f.lower().endswith(".png")] if os.path.isdir(cancer_image_dir) else []
        for img_name in cancer_images:
            mpath = os.path.join(cancer_mask_dir, img_name)
            m = patient_id_pattern.search(img_name)
            if os.path.isfile(mpath) and m:
                self.image_paths.append(os.path.join(cancer_image_dir, img_name))
                self.mask_paths.append(mpath)
                self.labels.append(1)
                self.patient_ids.append(m.group(1))

        not_cancer_images = [f for f in os.listdir(not_cancer_image_dir) if f.lower().endswith(".png")] if os.path.isdir(not_cancer_image_dir) else []
        for img_name in not_cancer_images:
            mpath = os.path.join(not_cancer_mask_dir, img_name)
            m = patient_id_pattern.search(img_name)
            if os.path.isfile(mpath) and m:
                self.image_paths.append(os.path.join(not_cancer_image_dir, img_name))
                self.mask_paths.append(mpath)
                self.labels.append(0)
                self.patient_ids.append(m.group(1))

        if mean is not None and std is not None:
            self.mean = list(mean)
            self.std = list(std)
        elif compute_stats:
            self.mean, self.std = self._compute_dataset_mean_std(max_samples=stats_sample_size)
        else:
            self.mean = [0.485, 0.456, 0.406]
            self.std  = [0.229, 0.224, 0.225]

        self.base_transform = A.Compose([
            A.Resize(224, 224, interpolation=cv2.INTER_LINEAR),
            A.Normalize(mean=self.mean, std=self.std),
            ToTensorV2(),
        ])

    def _compute_dataset_mean_std(self, max_samples=None):
        if len(self.image_paths) == 0:
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        indices = np.arange(len(self.image_paths))
        if max_samples is not None and max_samples < len(indices):
            np.random.shuffle(indices)
            indices = indices[:max_samples]

        n_pixels_total = 0
        channel_sum = np.zeros(3, dtype=np.float64)
        channel_sum_sq = np.zeros(3, dtype=np.float64)

        for idx in indices:
            img_path = self.image_paths[idx]
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            img_flat = img.reshape(-1, 3)
            n_pixels = img_flat.shape[0]
            n_pixels_total += n_pixels
            channel_sum += img_flat.sum(axis=0)
            channel_sum_sq += (img_flat ** 2).sum(axis=0)

        if n_pixels_total == 0:
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        mean = channel_sum / n_pixels_total
        var = (channel_sum_sq / n_pixels_total) - mean ** 2
        std = np.sqrt(np.maximum(var, 1e-12))
        return mean.tolist(), std.tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            return None, None

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = mask.astype(np.uint8)

        two_channel_mask = np.zeros((mask.shape[0], mask.shape[1], 2), dtype=np.float32)
        two_channel_mask[mask == 0, 0] = 1.0
        two_channel_mask[mask != 0, 1] = 1.0

        augmented = self.base_transform(image=img, mask=two_channel_mask)
        x = augmented["image"]
        y = augmented["mask"]
        if y.shape[0] != 2:
            # Defensive fallback
            resized = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
            two = np.zeros((224, 224, 2), dtype=np.float32)
            two[resized == 0, 0] = 1.0
            two[resized != 0, 1] = 1.0
            y = torch.from_numpy(two).permute(2, 0, 1)
        return x, y

In [ ]:
def create_stratified_subset(full_dataset: ProstateCancerDataset, ratio: float, seed: int) -> Subset:
    """
    Two-level stratification: patient -> class within patient.
    """
    from collections import defaultdict
    indices_by_patient_and_class = defaultdict(lambda: defaultdict(list))
    for i in range(len(full_dataset)):
        pid = full_dataset.patient_ids[i]
        lbl = full_dataset.labels[i]
        indices_by_patient_and_class[pid][lbl].append(i)

    subset_indices: List[int] = []
    g = torch.Generator().manual_seed(seed)

    for pid, class_groups in indices_by_patient_and_class.items():
        for lbl, idxs in class_groups.items():
            k = int(np.ceil(len(idxs) * ratio))
            perm = torch.randperm(len(idxs), generator=g).tolist()
            chosen = perm[:k]
            subset_indices.extend([idxs[j] for j in chosen])

    random.shuffle(subset_indices)
    return Subset(full_dataset, subset_indices)

In [ ]:
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None and x[0] is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch) if batch else None

In [ ]:
# -----------------------------
# Models
# -----------------------------
def get_model(architecture: str, encoder: str, decoder_dropout: float = 0.0):
    # LR finder: keep aux head disabled to avoid extra outputs; keep dropout only if you want.
    aux_params = None
    encoder_weights = "imagenet"

    arch = architecture.upper()
    if arch == "SWIN":
        # Your setup used smp.Unet with a swin encoder
        return smp.Unet(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                        activation=None, decoder_attention_type=None, aux_params=aux_params)
    if arch == "DEEPLABV3PLUS":
        return smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                                activation=None, decoder_attention_type=None, aux_params=aux_params)
    if arch == "UNET++":
        return smp.UnetPlusPlus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                                activation=None, decoder_attention_type=None, aux_params=aux_params)
    if arch == "FPN":
        return smp.FPN(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                       activation=None, decoder_attention_type=None, aux_params=aux_params)
    if arch == "SEGFORMER":
        return smp.Segformer(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                             activation=None, decoder_attention_type=None, aux_params=aux_params)
    if arch == "MANET":
        return smp.MAnet(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                         activation=None, decoder_attention_type=None, aux_params=aux_params)
    if arch == "DPT":
        return smp.DPT(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                       activation=None, decoder_attention_type=None, decoder_readout="ignore", aux_params=aux_params)
    if arch == "UPERNET":
        return smp.UPerNet(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=2,
                           activation=None, decoder_attention_type=None, aux_params=aux_params)
    raise ValueError(f"Unknown architecture: {architecture}")

In [ ]:
# Updated Search Space for BCEDiceHybridLossPaper
@dataclass(frozen=True)
class BCEDiceSearchSpace:
    alpha_min: float = 0.1
    alpha_max: float = 1.0
    beta_min: float = 0.1
    beta_max: float = 0.5
    gamma_min: float = 0.1
    gamma_max: float = 0.5
    gamma_log: bool = False  # Assuming uniform distribution for gamma

def latin_hypercube(n_samples: int, n_dims: int, seed: int = 24) -> np.ndarray:
    rng = np.random.default_rng(seed)
    X = np.zeros((n_samples, n_dims), dtype=np.float64)
    for d in range(n_dims):
        cut = np.linspace(0.0, 1.0, n_samples + 1)
        u = rng.random(n_samples)
        X[:, d] = cut[:-1] + u * (cut[1:] - cut[:-1])
        rng.shuffle(X[:, d])
    return X

def sample_bcedice_params(n: int, space: BCEDiceSearchSpace, seed: int) -> List[Dict[str, float]]:
    H = latin_hypercube(n, 3, seed)
    out = []
    for i in range(n):
        a_u, b_u, g_u = H[i]
        alpha = space.alpha_min + a_u * (space.alpha_max - space.alpha_min)
        beta  = space.beta_min  + b_u * (space.beta_max  - space.beta_min)
        if space.gamma_log:
            lo = math.log(space.gamma_min)
            hi = math.log(space.gamma_max)
            gamma = math.exp(lo + g_u * (hi - lo))
        else:
            gamma = space.gamma_min + g_u * (space.gamma_max - space.gamma_min)
        out.append({"alpha": float(alpha), "beta": float(beta), "gamma": float(gamma)})
    return out

In [ ]:
# -----------------------------
# Curve statistics
# -----------------------------
@dataclass
class CurveStats:
    smoothness_var: float
    largest_stable_lr: float
    divergence_lr: float
    min_loss: float
    stable_points: int

def _moving_average(x: np.ndarray, k: int = 5) -> np.ndarray:
    if len(x) < k:
        return x.copy()
    w = np.ones(k) / k
    return np.convolve(x, w, mode="same")

def compute_curve_stats(lrs: np.ndarray, losses: np.ndarray, skip_start: int = 10, skip_end: int = 5) -> CurveStats:
    """
    Two metrics requested:
      - smoothness: low variance of slope in stable region (lower is smoother)
      - largest stable LR before divergence
    Divergence rule (fast, reproducible):
      - find min_loss after skips
      - divergence when loss_smooth > min_loss * 4 OR loss is NaN/Inf
    """
    # guard
    lrs = np.asarray(lrs, dtype=np.float64)
    losses = np.asarray(losses, dtype=np.float64)

    n = len(lrs)
    lo = skip_start
    hi = max(lo + 1, n - skip_end)
    lrs_c = lrs[lo:hi]
    loss_c = losses[lo:hi]

    # remove non-finite
    finite_mask = np.isfinite(loss_c) & np.isfinite(lrs_c) & (lrs_c > 0)
    lrs_c = lrs_c[finite_mask]
    loss_c = loss_c[finite_mask]

    if len(lrs_c) < 10:
        return CurveStats(
            smoothness_var=float("inf"),
            largest_stable_lr=float("nan"),
            divergence_lr=float("nan"),
            min_loss=float("nan"),
            stable_points=0,
        )

    loss_s = _moving_average(loss_c, k=7)
    min_loss = float(np.min(loss_s))

    # divergence: first index where loss exceeds 4x min (heuristic)
    thresh = min_loss * 4.0
    div_idx = None
    for i, v in enumerate(loss_s):
        if not np.isfinite(v) or v > thresh:
            div_idx = i
            break

    if div_idx is None:
        div_idx = len(loss_s) - 1

    stable_lrs = lrs_c[:max(1, div_idx)]
    stable_loss = loss_s[:max(1, div_idx)]

    divergence_lr = float(lrs_c[div_idx]) if div_idx < len(lrs_c) else float(lrs_c[-1])
    largest_stable_lr = float(stable_lrs[-1])

    # smoothness: variance of slope wrt log-lr in stable region
    log_lr = np.log10(stable_lrs)
    # numeric derivative
    if len(log_lr) >= 5:
        slope = np.gradient(stable_loss, log_lr)
        smoothness_var = float(np.var(slope))
    else:
        smoothness_var = float("inf")

    return CurveStats(
        smoothness_var=smoothness_var,
        largest_stable_lr=largest_stable_lr,
        divergence_lr=divergence_lr,
        min_loss=min_loss,
        stable_points=int(len(stable_lrs)),
    )

In [ ]:
# -----------------------------
# Utilities
# -----------------------------
def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def fmt(x: float) -> str:
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "NA"
    if x >= 1e-2 and x < 1e3:
        return f"{x:.4f}"
    return f"{x:.2e}"

def latex_escape(s: str) -> str:
    return (s.replace("\\", "\\textbackslash{}")
             .replace("_", "\\_")
             .replace("%", "\\%")
             .replace("&", "\\&")
             .replace("#", "\\#")
             .replace("{", "\\{")
             .replace("}", "\\}")
             .replace("^", "\\^{}")
             .replace("~", "\\~{}"))

@dataclass
class RunRecord:
    architecture: str
    encoder: str
    alpha: float
    beta: float
    gamma: float
    # curve stats:
    smoothness_var: float
    largest_stable_lr: float
    divergence_lr: float
    min_loss: float
    stable_points: int
    # files:
    plot_path: str
    csv_path: str

In [ ]:
def run_lr_finder_once(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    criterion: torch.nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    end_lr: float,
    num_iter: int,
    amp_enabled: bool,
) -> Dict[str, np.ndarray]:
    """
    Returns: history dict with 'lr' and 'loss'.
    """
    lr_finder = LRFinder(model, optimizer, criterion, device=device)

    def _train_batch_patched(self, train_iter, accumulation_steps, non_blocking_transfer):
        self.model.train()
        try:
            batch_data = next(train_iter)
        except StopIteration:
            raise

        if batch_data is None:
            return float("nan")

        images, masks = batch_data
        if images is None:
            return float("nan")

        images = images.to(self.device, non_blocking=non_blocking_transfer)
        masks  = masks.to(self.device, non_blocking=non_blocking_transfer)

        self.optimizer.zero_grad(set_to_none=True)

        try:
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                outputs_raw = self.model(images)
                outputs = outputs_raw[0] if isinstance(outputs_raw, (tuple, list)) else outputs_raw
                loss = self.criterion(outputs, masks)

            if not torch.isfinite(loss):
                return float("nan")

            loss.backward()
            self.optimizer.step()

            loss_val = float(loss.detach().item())
            del outputs_raw, outputs, loss
            return loss_val

        except Exception as e:
            print(f"[LRFinder] ERROR during forward/backward: {e}")
            return float("nan")

    # patch LRFinder's internal train step
    lr_finder._train_batch = _train_batch_patched.__get__(lr_finder, LRFinder)

    # run LR range test
    lr_finder.range_test(train_loader, end_lr=end_lr, num_iter=num_iter, step_mode="exp")
    history = lr_finder.history
    lr_finder.reset()

    # make sure keys exist
    if history is None or ("lr" not in history) or ("loss" not in history):
        raise RuntimeError(f"LRFinder returned invalid history: {history}")

    return {
        "lr": np.array(history["lr"], dtype=np.float64),
        "loss": np.array(history["loss"], dtype=np.float64),
    }

In [ ]:
def plot_lr_curve(lrs: np.ndarray, losses: np.ndarray, title: str, out_png: Path, skip_start: int = 10, skip_end: int = 5):
    plt.figure()
    # Apply skips for visualization
    n = len(lrs)
    lo = skip_start
    hi = max(lo + 1, n - skip_end)
    lrs_c = lrs[lo:hi]
    loss_c = losses[lo:hi]

    plt.plot(np.log10(lrs_c), loss_c)
    plt.xlabel("log10(LR)")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

In [ ]:
# -----------------------------
# LaTeX report generation
# -----------------------------
def build_latex_report(
    records: List[RunRecord],
    out_dir: Path,
    meta: Dict[str, str],
    pdf_name: str = "report.pdf",
) -> Tuple[Path, Path]:
    """
    Creates report.tex and compiles to report.pdf (pdflatex).
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # group by architecture
    by_arch: Dict[str, List[RunRecord]] = {}
    for r in records:
        by_arch.setdefault(r.architecture, []).append(r)

    tex_lines = []
    tex_lines.append(r"\documentclass[11pt]{article}")
    tex_lines.append(r"\usepackage[a4paper,margin=1.8cm]{geometry}")
    tex_lines.append(r"\usepackage{graphicx}")
    tex_lines.append(r"\usepackage{float}")
    tex_lines.append(r"\usepackage{booktabs}")
    tex_lines.append(r"\usepackage{longtable}")
    tex_lines.append(r"\usepackage{hyperref}")
    tex_lines.append(r"\usepackage{caption}")
    tex_lines.append(r"\captionsetup{font=small,labelfont=bf}")
    tex_lines.append(r"\begin{document}")
    tex_lines.append(r"\title{LR Finder Screening with Latin Hypercube Sampling over BCE Parameters}")
    tex_lines.append(rf"\date{{{latex_escape(meta.get('timestamp',''))}}}")
    tex_lines.append(r"\maketitle")

    tex_lines.append(r"\section*{Protocol}")
    tex_lines.append(r"\begin{itemize}")
    for k, v in meta.items():
        if k == "timestamp":
            continue
        tex_lines.append(rf"\item \textbf{{{latex_escape(k)}}}: {latex_escape(str(v))}")
    tex_lines.append(r"\end{itemize}")
    tex_lines.append(r"\clearpage")

    for arch, recs in by_arch.items():
        tex_lines.append(rf"\section{{{latex_escape(arch)}}}")
        enc_name = latex_escape(recs[0].encoder) if recs else "NA"
        tex_lines.append(rf"\noindent\textbf{{Encoder:}} {enc_name}\\")
        tex_lines.append(r"\medskip")

        # table
        tex_lines.append(r"\begin{longtable}{lllrrrrr}")
        tex_lines.append(r"\caption{Curve statistics for " + latex_escape(arch) + r"}\\")
        tex_lines.append(r"\toprule")
        tex_lines.append(r"$alpha$ & $\beta$ & $\gamma$ & SmoothVar & StableLR & DivLR & MinLoss & StablePts \\")
        tex_lines.append(r"\midrule")
        tex_lines.append(r"\endfirsthead")
        tex_lines.append(r"\toprule")
        tex_lines.append(r"$alpha$ & $\beta$ & $\gamma$ & SmoothVar & StableLR & DivLR & MinLoss & StablePts \\")
        tex_lines.append(r"\midrule")
        tex_lines.append(r"\endhead")

        # stable sorting: smoother first, then larger stable lr
        recs_sorted = sorted(recs, key=lambda r: (r.smoothness_var, -r.largest_stable_lr))
        for r in recs_sorted:
          tex_lines.append(
              rf"{r.alpha:.3f} & {r.beta:.3f} & {r.gamma:.3f} & "
              rf"{fmt(r.smoothness_var)} & {fmt(r.largest_stable_lr)} & "
              rf"{fmt(r.divergence_lr)} & {fmt(r.min_loss)} & {r.stable_points} \\"
              )

        tex_lines.append(r"\bottomrule")
        tex_lines.append(r"\end{longtable}")
        tex_lines.append(r"\clearpage")

        # figures
        for r in recs_sorted:
            tex_lines.append(r"\begin{figure}[H]")
            tex_lines.append(r"\centering")
            rel = Path(r.plot_path).relative_to(out_dir)
            tex_lines.append(rf"\includegraphics[width=\linewidth]{{{latex_escape(str(rel))}}}")
            cap = (
                f"Architecture={r.architecture}. "
                f"BCE(alpha={r.alpha:.3f}, beta={r.beta:.3f}, gamma={r.gamma:.3f}). "
                f"SmoothVar={fmt(r.smoothness_var)}, "
                f"StableLR={fmt(r.largest_stable_lr)}, "
                f"DivLR={fmt(r.divergence_lr)}."
            )

            tex_lines.append(rf"\caption{{{latex_escape(cap)}}}")
            tex_lines.append(r"\end{figure}")
            tex_lines.append(r"\clearpage")

    tex_lines.append(r"\end{document}")

    tex_path = out_dir / "report.tex"
    tex_path.write_text("\n".join(tex_lines), encoding="utf-8")

    # compile
    import subprocess
    cmd = ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", str(tex_path.name)]
    for _ in range(2):
        subprocess.run(cmd, cwd=str(out_dir), check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    pdf_path = out_dir / "report.pdf"
    return tex_path, pdf_path

In [ ]:
def get_database(base_data_dir, train_cancer_image_dir, dataset_dir):
  print(f"\n{'='*25} Starting Main Training Process {'='*25}")

  fold_zip_filename = f'MASTER_SET_1.zip'
  fold_zip_path = os.path.join(dataset_dir, fold_zip_filename)

  # --- Extract Dataset ---
  print(f"Extracting Fold...")
  if not os.path.exists(fold_zip_path):
    print(f"Zip not found: {fold_zip_path}. Skip.");
  try:
      if os.path.exists(base_data_dir):
        shutil.rmtree(base_data_dir)
      os.makedirs(base_data_dir, exist_ok=True);

      with zipfile.ZipFile(fold_zip_path,'r') as z:
        z.extractall(base_data_dir)

      print("Extracted. Verifying...");

      if not os.path.isdir(train_cancer_image_dir) or not os.listdir(train_cancer_image_dir):
        raise RuntimeError("Verify failed")

      print("Verified.")

  except Exception as e:
    print(f"Extract Err: {e}. Skip.");

In [ ]:
# -----------------------------
# Helpers for run IDs / names
# -----------------------------
def make_run_tag(arch, enc, w, d, g, idx):
    # idx is 1-based, fixed-width for sorting
    return f"{arch}_a{w:.3f}_b{d:.3f}_g{g:.3f}__{idx:03d}"

def print_arch_banner(arch, enc):
    print("\n" + "="*80)
    print(f"[ARCH] {arch} | [ENC] {enc}")
    print("="*80)

In [ ]:
dataset_dir = change_paths('IA_MEDICA_SAMPLES/DIAGSET/VAHADANE')

output_dir_drive = change_paths('LR_FINDER_REPORTS/DIAGSET/VAHADANE')

output_dir = '/content/reports/LR_FINDER_REPORTS'
base_data_dir = '/content/dataset'
seed = 24
batch_size = 8
workers = 2
subset_ratio = 0.2
use_subset = True
stats_sample_size = None                # 20000
n_lhs = 12
end_lr = 1e-1
num_iter = 100
amp = False

In [ ]:
out_dir = Path(output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

seed_everything(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Dataset paths (match your folder structure)
base = Path(base_data_dir)
train_cancer_image_dir = base/'TRAIN/CANCER'
train_cancer_mask_dir  = base/'TRAIN/CANCER_MASK'
train_not_cancer_image_dir = base/'TRAIN/NOT_CANCER'
train_not_cancer_mask_dir  = base/'TRAIN/NOT_CANCER_MASK'

get_database(base_data_dir, train_cancer_image_dir, dataset_dir)

In [ ]:
# Build TRAIN dataset stats (scientifically: stats from TRAIN only)
full_train_ds = ProstateCancerDataset(
    str(train_cancer_image_dir), str(train_cancer_mask_dir),
    str(train_not_cancer_image_dir), str(train_not_cancer_mask_dir),
    compute_stats=True, stats_sample_size=stats_sample_size,
)
train_mean = full_train_ds.mean
train_std  = full_train_ds.std
print("Computed TRAIN mean/std:", train_mean, train_std)

if use_subset and subset_ratio < 1.0:
    train_ds = create_stratified_subset(full_train_ds, subset_ratio, seed=seed)
else:
    train_ds = full_train_ds

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    num_workers=workers, pin_memory=True, drop_last=True,
    persistent_workers=(workers > 0),
    prefetch_factor=2 if workers > 0 else None,
    collate_fn=collate_fn
)

In [ ]:
# Architectures / encoders (as requested)
LIST_ARCH = ['SWIN','DEEPLABV3PLUS','UNET++','FPN','SEGFORMER','MANET','DPT','UPERNET']
LIST_ENCODER = [
    'tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k',
    'tu-resnest101e',
    'efficientnet-b7',
    'senet154',
    'mit_b5',
    'resnet152',
    'tu-vit_large_patch16_224.augreg_in21k_ft_in1k',
    'tu-hiera_large_224',
]


In [ ]:
# -----------------------------
# Experiment plan (ONE source of truth)
# -----------------------------
assert len(LIST_ARCH) == len(LIST_ENCODER), "LIST_ARCH and LIST_ENCODER must match length"

# Define the search space
space = BCEDiceSearchSpace(
    alpha_min=0.1, alpha_max=1.0,
    beta_min=0.1, beta_max=0.5,
    gamma_min=0.1, gamma_max=0.5,
    gamma_log=False
)

lhs_samples = sample_bcedice_params(n=n_lhs, space=space, seed=seed)

(out_dir / "LHS_SAMPLES.json").write_text(json.dumps(lhs_samples, indent=2), encoding="utf-8")

N_ARCH = len(LIST_ARCH)
N_LHS  = len(lhs_samples)
TOTAL_RUNS = N_ARCH * N_LHS

print(f"\n[INFO] Total planned runs = {N_ARCH} architectures × {N_LHS} LHS samples = {TOTAL_RUNS}\n")

# tqdm: global + per-arch
global_pbar = tqdm(total=TOTAL_RUNS, desc="GLOBAL LR-FINDER", unit="run", dynamic_ncols=True)

records: List[RunRecord] = []
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

completed_ok = 0
failed_runs = 0

for arch, enc in zip(LIST_ARCH, LIST_ENCODER):
    print_arch_banner(arch, enc)

    arch_pbar = tqdm(total=N_LHS, desc=f"{arch}", unit="run", dynamic_ncols=True, leave=False)

    for j, p in enumerate(lhs_samples, start=1):
        alpha, beta, gamma = p["alpha"], p["beta"], p["gamma"]
        tag = f"Loss_alpha_{alpha:.3f}_beta_{beta:.3f}_gamma_{gamma:.3f}"

        run_dir = out_dir / arch / f"{j:03d}"
        run_dir.mkdir(parents=True, exist_ok=True)

        # ---- progress postfix BEFORE running ----
        remaining = TOTAL_RUNS - global_pbar.n
        global_pbar.set_postfix_str(f"remaining={remaining} ok={completed_ok} fail={failed_runs}")

        # model + optimizer
        model = get_model(arch, enc).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-8, weight_decay=1e-4)

        criterion = BCEDiceHybridLossPaper(alpha=alpha, beta=beta, gamma=gamma)

        try:
            hist = run_lr_finder_once(
                model=model,
                optimizer=optimizer,
                criterion=criterion,
                train_loader=train_loader,
                device=device,
                end_lr=end_lr,
                num_iter=num_iter,
                amp_enabled=amp and (device.type == "cuda"),
            )

            lrs = hist["lr"]
            losses = hist["loss"]

            stats = compute_curve_stats(lrs, losses, skip_start=10, skip_end=5)

            # save csv
            csv_path = run_dir / f"{tag}.csv"
            pd.DataFrame({"lr": lrs, "loss": losses}).to_csv(csv_path, index=False)

            # save plot
            png_path = run_dir / f"{tag}.png"
            title = f"{arch} | a={alpha:.3f}, b={beta:.3f}, g={gamma:.3f}"
            plot_lr_curve(lrs, losses, title=title, out_png=png_path, skip_start=10, skip_end=5)

            records.append(RunRecord(
                architecture=arch,
                encoder=enc,
                alpha=alpha,
                beta=beta,
                gamma=gamma,
                smoothness_var=stats.smoothness_var,
                largest_stable_lr=stats.largest_stable_lr,
                divergence_lr=stats.divergence_lr,
                min_loss=stats.min_loss,
                stable_points=stats.stable_points,
                plot_path=str(png_path),
                csv_path=str(csv_path),
            ))

            completed_ok += 1
            print(f"[OK] {tag} | SmoothVar={fmt(stats.smoothness_var)} | StableLR={fmt(stats.largest_stable_lr)} | DivLR={fmt(stats.divergence_lr)}")

        except Exception as e:
            failed_runs += 1
            print(f"[FAIL] {tag} | {e}")

        finally:
            # always cleanup and advance progress bars
            del model, optimizer, criterion
            gc.collect()
            clear_gpu()

            global_pbar.update(1)
            arch_pbar.update(1)

    arch_pbar.close()

    # after each architecture: save summary
    arch_df = pd.DataFrame([asdict(r) for r in records if r.architecture == arch])
    if len(arch_df) > 0:
        arch_df.to_csv(out_dir / arch / f"SUMMARY_{arch}.csv", index=False)

global_pbar.close()

# global summary
all_df = pd.DataFrame([asdict(r) for r in records])
all_df.to_csv(out_dir / "SUMMARY_ALL.csv", index=False)

In [ ]:
meta = {
    "timestamp": timestamp,
    "seed": str(seed),
    "device": str(device),
    "batch_size": str(batch_size),
    "workers": str(workers),
    "subset_ratio": str(subset_ratio) if use_subset else "1.0",
    "lhs_n": str(n_lhs),
    "BCE_alpha_range": f"[{space.alpha_min}, {space.alpha_max}]",
    "BCE_beta_range": f"[{space.beta_min}, {space.beta_max}]",
    "BCE_gamma_range": f"[{space.gamma_min}, {space.gamma_max}] " + ("(log-uniform)" if space.gamma_log else "(uniform)"),
    "optimizer": "torch.optim.AdamW (lr initialized 1e-8 for range test)",
    "lr_finder": f"torch_lr_finder.LRFinder | end_lr={end_lr} | num_iter={num_iter} | step_mode=exp",
    "amp": str(bool(amp and (str(device) == "cuda"))),
    "metrics": "smoothness_var(var(dLoss/dlogLR) over stable region), largest_stable_lr(before divergence)",
    "divergence_rule": "loss_smooth > 4 * min_loss (after skips) or non-finite",
    "skips": "skip_start=10, skip_end=5",
    "MEAN_full_db": f"{train_mean[0]:.3f}, {train_mean[1]:.3f}, {train_mean[2]:.3f}",
    "STD_full_db": f"{train_std[0]:.3f}, {train_std[1]:.3f}, {train_std[2]:.3f}",
}

print("\nBuilding LaTeX report...")
tex_path, pdf_path = build_latex_report(records, out_dir, meta=meta)
print("Wrote:", tex_path)
print("PDF :", pdf_path)

In [ ]:
from pathlib import Path
import shutil

def copy_out_dir_to_final(out_dir: str | Path, final_report_path: str | Path, overwrite: bool = True) -> Path:
    """
    Copy the entire contents of `out_dir` into `final_report_path`.

    - If `overwrite=True`, it will delete `final_report_path` first (if it exists),
      then copy everything fresh (recommended for reproducibility).
    - If `overwrite=False`, it will merge contents (may overwrite files with same names).

    Returns:
        Path to the final_report_path.
    """
    src = Path(out_dir)
    dst = Path(final_report_path)

    if not src.exists() or not src.is_dir():
        raise FileNotFoundError(f"out_dir does not exist or is not a directory: {src}")

    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        if overwrite:
            shutil.rmtree(dst)
        else:
            # Merge-copy (Python >= 3.8 supports dirs_exist_ok)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            return dst

    # Fresh copy
    shutil.copytree(src, dst)
    return dst

In [ ]:
# --- Helper Functions ---
def get_formatted_datetime_string():
  now = datetime.now()
  return now.strftime("%d_%m_%Y_%H_%M_%S")

In [ ]:
os.makedirs(output_dir, exist_ok=True)
dst = os.path.join(output_dir_drive, get_formatted_datetime_string())
shutil.copytree(output_dir, dst)